In [1]:
import os
import glob
import rasterio
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pdb
from IPython.display import display
import ee
import os

In [2]:
import ee
import geemap
service_account = ' boba-account@boba-430314.iam.gserviceaccount.com '
credentials = ee.ServiceAccountCredentials(service_account,'C:/Users/arj26323/OneDrive - University of Georgia/Documents/GEE AUTH/boba-430314-a19be859bfca.json')

ee.Initialize(credentials)

print(geemap.__version__)

0.34.1


In [3]:
mml_path = "F:/Hotspots/Project shapefiles/mml.shp"
mml = geemap.shp_to_ee(mml_path)

GeoPandas is required to perform reprojection of the data. See https://geopandas.org/install.html


In [4]:
# mml.getInfo()

In [5]:
##FOR 10m DEM:
def demExtraction(image):
    feature = image.sampleRegions(
        collection = fc_all,
        scale = 10 
    )
    return feature

dem = ee.Image('USGS/3DEP/10m') 
elev = dem.select('elevation')
slope = ee.Terrain.slope(dem)

In [6]:
Map = geemap.Map(center=[31.539096,-81.422318], zoom=10)

elevVis = {
    'min': 0,
    'max': 500,
    'palette': ['#3288bd', '#fee08b', '#d53e4f'] # Blue (low) to Red (high)
}

slopeVis = {
    'min': 0,
    'max': 10,
    'palette': ['#3288bd', '#fee08b', '#d53e4f'] # Blue (flat) to Red (steep)
}

Map.addLayer(elev, elevVis, 'elevation')
Map.addLayer(slope, slopeVis, 'slope')


In [7]:
# Map

In [18]:
fc = mml
# 3. Extract slope values along the lines
# Using reduceRegions to get the average slope for each polyline

slope_data = slope.reduceRegions( collection=fc, 
                                  reducer=ee.Reducer.mean(), # Or ee.Reducer.max() if you want the highest slope point 
                                  scale=30 )
# 4. Flag/Filter regions above a certain slope (e.g., 15 degrees)

threshold = 0
high_slope_lines = slope_data.filter(ee.Filter.gt('mean', threshold))
# Display on map

Map.add_layer(high_slope_lines, {'color': 'yellow'}, 'High Slope Lines')
# Map.center_object(high_slope_lines)

Map

Map(bottom=53697.0, center=[31.60403659820493, -81.55700683593751], controls=(WidgetControl(options=['position…

In [49]:
fc = mml 
fc = fc.map(lambda f: f.set('total_length_m', f.length(maxError=1)))
high_slope_mask = slope.gt(threshold)

threshold = 10

def get_high_slope_length_fixed(feature):
    # 1. Create a buffer
    buffered = feature.buffer(30, maxError=1)
    
    # 2. Extract only the high-slope pixels as a geometry
    # .selfMask() ensures we only get the steep pixels
    steep_geometry = high_slope_mask.updateMask(high_slope_mask).clip(buffered).reduceToVectors(
        geometry=buffered.geometry(),
        scale=30,
        geometryType='polygon'
    )
    
    # 3. Intersect the original line with the steep polygons
    # This gives you the specific segments of the line that are steep
    steep_segments = feature.intersection(steep_geometry.geometry(), maxError=1)
    
    # 4. Get the actual length of these specific segments
    return feature.set('high_slope_length_m', steep_segments.length(maxError=1))

# Now run this, and you will get the actual geometry length
results = fc.map(get_high_slope_length_fixed)


def add_percentage(feature):
    total = ee.Number(feature.get('total_length_m'))
    steep = ee.Number(feature.get('high_slope_length_m'))
    # Avoid division by zero
    percent = steep.divide(total).multiply(100)
    return feature.set('percent_high_slope', percent)

final_results = results.map(add_percentage)

In [50]:
# final_results

In [6]:
'''# 5. Export to CSV for final analysis
geemap.ee_export_vector(
    final_results, 
    filename="C:/Downloads/slopesandlengths.csv", 
    verbose=True
)'''

'# 5. Export to CSV for final analysis\ngeemap.ee_export_vector(\n    final_results, \n    filename="C:/Downloads/slopesandlengths.csv", \n    verbose=True\n)'

In [30]:
buffered_fc = fc.map(lambda feature: feature.buffer(30)) ##buffering the lines so that they can be summarized easily

slope_threshold = 15 ##15 degrees - is this a high slope for marsh migration?
high_slope_mask = slope.gt(slope_threshold)


slope_segments = slope.reduceRegions(
    collection=buffered_fc,
    reducer=ee.Reducer.mean(), # Or max, or even a list of all pixel values
    scale=30
)



# 2. Extract the high-slope areas as polygons
# This turns the "high slope" pixels into vector shapes
high_slope_polygons = high_slope_mask.selfMask().reduceToVectors(
    geometry=fc.geometry().bounds(),
    scale=30,
    geometryType='polygon'
)

# 3. Intersect your original lines with these polygons
# This leaves you only with the portions of your lines inside high-slope zones
high_slope_segments = fc.map(lambda feature: 
    feature.intersection(high_slope_polygons.geometry(), ee.ErrorMargin(1))
)

# 4. Calculate length of the resulting segments
# Note: You may need to flatten the output if segments are broken into pieces
def add_length(feature):
    return feature.set('segment_length_m', feature.length())

segments_with_length = high_slope_segments.map(add_length)

NameError: name 'segments_with_length' is not defined

In [31]:
segments_with_length

# Grouping by 'line_id' to sum segment_length_m
stats = segments_with_length.reduceColumns(
    reducer=ee.Reducer.sum().group(
        groupField=0,
        groupName='line_id'
    ),
    selectors=['line_id', 'segment_length_m']
)

print(stats.getInfo())

{'groups': []}


EEException: Image.reduceToVectors: Too many pixels in the region. Found 14777375, but maxPixels allows only 10000000.
Ensure that you are not aggregating at a higher resolution than you intended; that is a frequent cause of this error. If not, then you may set the 'maxPixels' argument to a limit suitable for your computation; set 'bestEffort' to true to aggregate at whatever scale results in 'maxPixels' total pixels; or both.